# Prepare the 300-case human–LLM annotation evaluation data

This notebook converts three human annotation exports and the corresponding final-pipeline LLM annotations into stable relational tables for evaluation. It does not match entities or construct a human consensus; ad, face, and group IDs remain coder-local.

The outputs retain original categorical labels, normalize only documented schema differences, and put all bounding boxes on a common 0–1 coordinate scale. Interaction telemetry and navigation histories are excluded from the analytical tables because they are not annotation outcomes; they remain available in the immutable source JSONL files.

In [ ]:
from __future__ import annotations

import hashlib
import json
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from thesis_tables import export_quarto_table

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

NOTEBOOK_DIR = Path.cwd().resolve()
EXPECTED_CASES = 300

INPUT_DIR = (NOTEBOOK_DIR / "../../data/annotations/llm_evaluation_300").resolve()
HUMAN_PATHS = {
    "A": INPUT_DIR / "A_economist_decade_face_count_stratified_600_v3_64597498_2026-09-11.jsonl",
    "B": INPUT_DIR / "B_economist_decade_face_count_stratified_600_v3_60109424_2026-09-11.jsonl",
    "C": INPUT_DIR / "C_economist_decade_face_count_stratified_600_v3_25867936_2026-09-11.jsonl",
}
LLM_PATH = INPUT_DIR / "qwen_full_pages_joined_v1_snapshot_20260820_121308.jsonl"
OUTPUT_DIR = (NOTEBOOK_DIR / "../../data/processed/llm_evaluation_300").resolve()

for path in [*HUMAN_PATHS.values(), LLM_PATH]:
    assert path.is_file(), f"Missing input: {path}"

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Output directory:   {OUTPUT_DIR}")

## Load and prove the common evaluation set

JSONL is parsed line by line so malformed records can be localized. The evaluation universe is defined by human `image_index` values 0–299; all three human files must contain the same ordered IDs, and the LLM snapshot must contain one successful result for every target ID.

In [ ]:
def read_jsonl(path: Path) -> tuple[list[dict[str, Any]], list[dict[str, Any]], int]:
    rows: list[dict[str, Any]] = []
    errors: list[dict[str, Any]] = []
    blank_lines = 0
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                blank_lines += 1
                continue
            try:
                value = json.loads(line)
                if not isinstance(value, dict):
                    raise TypeError(f"top-level value is {type(value).__name__}, expected object")
                value["_source_line"] = line_number
                rows.append(value)
            except Exception as exc:
                errors.append({"path": str(path), "line": line_number, "error": repr(exc)})
    return rows, errors, blank_lines


human_raw: dict[str, list[dict[str, Any]]] = {}
parse_findings: list[dict[str, Any]] = []
load_rows: list[dict[str, Any]] = []

for source_id, path in HUMAN_PATHS.items():
    rows, errors, blanks = read_jsonl(path)
    human_raw[source_id] = rows
    parse_findings.extend(errors)
    load_rows.append({
        "source_id": source_id,
        "source_type": "human",
        "path": str(path),
        "rows": len(rows),
        "blank_lines": blanks,
        "parse_errors": len(errors),
        "unique_image_ids": len({str(row.get("image_id")) for row in rows}),
    })

llm_raw, llm_errors, llm_blanks = read_jsonl(LLM_PATH)
parse_findings.extend(llm_errors)
load_rows.append({
    "source_id": "llm_qwen_final_standalone_v1",
    "source_type": "llm",
    "path": str(LLM_PATH),
    "rows": len(llm_raw),
    "blank_lines": llm_blanks,
    "parse_errors": len(llm_errors),
    "unique_image_ids": len({str(row.get("image_id")) for row in llm_raw}),
})

assert not parse_findings, pd.DataFrame(parse_findings).to_string(index=False)

human_selected: dict[str, list[dict[str, Any]]] = {}
for source_id, rows in human_raw.items():
    selected = sorted(
        [row for row in rows if isinstance(row.get("image_index"), int) and 0 <= row["image_index"] < EXPECTED_CASES],
        key=lambda row: row["image_index"],
    )
    assert len(selected) == EXPECTED_CASES, f"{source_id}: expected {EXPECTED_CASES} cases, found {len(selected)}"
    assert [row["image_index"] for row in selected] == list(range(EXPECTED_CASES))
    assert len({str(row["image_id"]) for row in selected}) == EXPECTED_CASES
    human_selected[source_id] = selected

reference_source = "B"
reference_rows = human_selected[reference_source]
target_ids = [str(row["image_id"]) for row in reference_rows]
target_id_set = set(target_ids)
index_by_id = {str(row["image_id"]): int(row["image_index"]) for row in reference_rows}

for source_id, rows in human_selected.items():
    assert [str(row["image_id"]) for row in rows] == target_ids, f"{source_id}: ordered case IDs differ"
    assert [str(row["filename"]) for row in rows] == [str(row["filename"]) for row in reference_rows]

llm_target_candidates = [row for row in llm_raw if str(row.get("image_id")) in target_id_set]
llm_counts = Counter(str(row.get("image_id")) for row in llm_target_candidates)
assert set(llm_counts) == target_id_set
assert all(count == 1 for count in llm_counts.values()), "Duplicate LLM records among target cases"
assert all(row.get("ok") is True and isinstance(row.get("annotation"), dict) for row in llm_target_candidates)
llm_selected = sorted(llm_target_candidates, key=lambda row: index_by_id[str(row["image_id"])])

load_audit = pd.DataFrame(load_rows)
display(load_audit)
print(f"Common ordered cases: {len(target_ids)}")
print(f"LLM coverage: {len(llm_selected)}/{EXPECTED_CASES}")

## Normalize schemas without changing labels

The human app stores boxes on a 0–1 scale (`bbox`, `face_bbox`) and the LLM stores integer-like coordinates on a 0–1000 scale (`bbox_1000`, `face_bbox_1000`). The LLM also calls its ad identifier `advertisement_id`; the common tables expose this as `ad_id`. Original box values and source scale are retained alongside normalized geometry.

`depiction_type_resolved` is an added convenience field for people: person-level values take precedence and otherwise inherit the ad-level depiction type. The original person-level `depiction_type` is not overwritten.

In [ ]:
def json_text(value: Any) -> str | None:
    if value is None:
        return None
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def integer_or_none(value: Any) -> int | None:
    if value is None or value == "":
        return None
    return int(value)


def geometry_fields(entity: dict[str, Any], *, face: bool = False) -> dict[str, Any]:
    unit_key = "face_bbox" if face else "bbox"
    thousand_key = "face_bbox_1000" if face else "bbox_1000"
    if entity.get(unit_key) is not None:
        raw = entity[unit_key]
        scale = "0_1"
        divisor = 1.0
    elif entity.get(thousand_key) is not None:
        raw = entity[thousand_key]
        scale = "0_1000"
        divisor = 1000.0
    else:
        raw = None
        scale = None
        divisor = None

    result = {
        "bbox_raw_json": json_text(raw),
        "bbox_source_scale": scale,
        "bbox_x1": np.nan,
        "bbox_y1": np.nan,
        "bbox_x2": np.nan,
        "bbox_y2": np.nan,
        "bbox_width": np.nan,
        "bbox_height": np.nan,
        "bbox_area": np.nan,
        "bbox_center_x": np.nan,
        "bbox_center_y": np.nan,
    }
    if raw is None:
        return result
    if not isinstance(raw, list) or len(raw) != 4:
        raise ValueError(f"Invalid box container: {raw!r}")
    x1, y1, x2, y2 = [float(value) / divisor for value in raw]
    result.update({
        "bbox_x1": x1,
        "bbox_y1": y1,
        "bbox_x2": x2,
        "bbox_y2": y2,
        "bbox_width": x2 - x1,
        "bbox_height": y2 - y1,
        "bbox_area": (x2 - x1) * (y2 - y1),
        "bbox_center_x": (x1 + x2) / 2,
        "bbox_center_y": (y1 + y2) / 2,
    })
    return result


reference_metadata = {}
for row in reference_rows:
    annotation = row["annotation"]
    image = annotation.get("image") or {}
    reference_metadata[str(row["image_id"])] = {
        "filename": str(row["filename"]),
        "page_type": image.get("page_type"),
        **(image.get("metadata") or {}),
    }

source_records: list[tuple[str, str, dict[str, Any]]] = []
for source_id, rows in human_selected.items():
    source_records.extend((source_id, "human", row) for row in rows)
source_records.extend(("llm_qwen_final_standalone_v1", "llm", row) for row in llm_selected)

case_rows: list[dict[str, Any]] = []
ad_rows: list[dict[str, Any]] = []
person_rows: list[dict[str, Any]] = []
group_rows: list[dict[str, Any]] = []

for source_id, source_type, record in source_records:
    image_id = str(record["image_id"])
    annotation = record["annotation"]
    page = annotation.get("page") or {}
    ads = annotation.get("advertisements") or []
    metadata = reference_metadata[image_id]
    declared_ad_count = integer_or_none(page.get("qualifying_ad_count", annotation.get("qualifying_ad_count")))
    case_key = f"{source_id}::{image_id}"
    case_rows.append({
        "case_key": case_key,
        "source_id": source_id,
        "source_type": source_type,
        "image_index": index_by_id[image_id],
        "image_id": image_id,
        "filename": str(record.get("filename") or metadata["filename"]),
        "year": integer_or_none(metadata.get("year")) or int(image_id[:4]),
        "decade": integer_or_none(metadata.get("decade")) or (int(image_id[:4]) // 10) * 10,
        "page_type": metadata.get("page_type"),
        "sampling_face_count": integer_or_none(metadata.get("face_count")),
        "sampling_face_count_bucket": metadata.get("face_count_bucket"),
        "annotation_status": record.get("annotation_status") or ("complete" if record.get("ok") is True else "failed"),
        "annotation_schema_version": annotation.get("schema_version"),
        "source_line": record["_source_line"],
        "declared_qualifying_ad_count": declared_ad_count,
        "observed_ad_rows": len(ads),
        "page_unique_face_count": integer_or_none(page.get("unique_face_count")),
        "page_duplicate_faces_present": page.get("duplicate_faces_present"),
        "no_qualifying_ad_reason": page.get("no_qualifying_ad_reason", annotation.get("no_qualifying_ad_reason")),
        "page_review_flags_json": json_text(annotation.get("review_flags")),
        "assignment_code": record.get("assignment_code"),
        "assignee_name": record.get("assignee_name"),
        "revision": record.get("revision"),
        "server_completed_at": record.get("server_completed_at"),
        "llm_route": record.get("route"),
        "llm_cohort": record.get("cohort"),
    })

    for ad_position, ad in enumerate(ads):
        raw_ad_id = ad.get("ad_id") if ad.get("ad_id") is not None else ad.get("advertisement_id")
        if raw_ad_id is None:
            raise ValueError(f"Missing ad ID: {source_id} / {image_id} / position {ad_position}")
        ad_id = str(raw_ad_id)
        ad_key = f"{case_key}::{ad_id}"
        ad_rows.append({
            "ad_key": ad_key,
            "case_key": case_key,
            "source_id": source_id,
            "source_type": source_type,
            "image_index": index_by_id[image_id],
            "image_id": image_id,
            "ad_position": ad_position,
            "ad_id": ad_id,
            "original_id_field": "ad_id" if ad.get("ad_id") is not None else "advertisement_id",
            "extent": ad.get("extent"),
            "depiction_type": ad.get("depiction_type"),
            "face_depiction_count_band": ad.get("face_depiction_count_band"),
            "unique_face_count": integer_or_none(ad.get("unique_face_count")),
            "duplicate_faces_present": ad.get("duplicate_faces_present"),
            "has_outstanding_individuals": ad.get("has_outstanding_individuals"),
            "observed_person_rows": len(ad.get("people") or []),
            "observed_group_rows": len(ad.get("groups") or []),
            "bbox_source": ad.get("bbox_source"),
            "ad_category": ad.get("ad_category"),
            "brand_or_advertiser": ad.get("brand_or_advertiser"),
            "confidence": ad.get("confidence"),
            "review_flags_json": json_text(ad.get("review_flags")),
            **geometry_fields(ad),
        })

        for person_position, person in enumerate(ad.get("people") or []):
            person_id = str(person.get("person_id"))
            person_rows.append({
                "person_key": f"{ad_key}::{person_id}",
                "ad_key": ad_key,
                "case_key": case_key,
                "source_id": source_id,
                "source_type": source_type,
                "image_index": index_by_id[image_id],
                "image_id": image_id,
                "ad_id": ad_id,
                "ad_position": ad_position,
                "person_position": person_position,
                "person_id": person_id,
                "annotation_role": person.get("annotation_role"),
                "depiction_type": person.get("depiction_type"),
                "depiction_type_resolved": person.get("depiction_type") or ad.get("depiction_type"),
                "perceived_age": person.get("perceived_age"),
                "perceived_gender_presentation": person.get("perceived_gender_presentation"),
                "face_expression_legibility": person.get("face_expression_legibility"),
                "face_orientation": person.get("face_orientation"),
                "gaze_target": person.get("gaze_target"),
                "gaze_target_person_id": person.get("gaze_target_person_id"),
                "gaze_target_person_unboxed": person.get("gaze_target_person_unboxed"),
                "gaze_target_object_ref": person.get("gaze_target_object_ref"),
                "mouth_covered": person.get("mouth_covered"),
                "mouth_covering": person.get("mouth_covering"),
                "mouth_covering_other_text": person.get("mouth_covering_other_text"),
                "smile_present": person.get("smile_present"),
                "smile_intensity": person.get("smile_intensity"),
                "duplicate_of_person_id": person.get("duplicate_of_person_id"),
                "duplicate_person_ids_json": json_text(person.get("duplicate_person_ids")),
                "is_duplicate_copy": person.get("duplicate_of_person_id") is not None,
                "confidence": person.get("confidence"),
                "prominence_reason": person.get("prominence_reason"),
                "review_flags_json": json_text(person.get("review_flags")),
                **geometry_fields(person, face=True),
            })

        for group_position, group in enumerate(ad.get("groups") or []):
            group_id = str(group.get("group_id"))
            group_rows.append({
                "group_key": f"{ad_key}::{group_id}",
                "ad_key": ad_key,
                "case_key": case_key,
                "source_id": source_id,
                "source_type": source_type,
                "image_index": index_by_id[image_id],
                "image_id": image_id,
                "ad_id": ad_id,
                "ad_position": ad_position,
                "group_position": group_position,
                "group_id": group_id,
                "group_type": group.get("group_type"),
                "age_composition": group.get("age_composition"),
                "gender_presentation_composition": group.get("gender_presentation_composition"),
                "expression_legibility_distribution": group.get("expression_legibility_distribution"),
                "dominant_gaze": group.get("dominant_gaze"),
                "smile_prevalence": group.get("smile_prevalence"),
                "dominant_smile_intensity": group.get("dominant_smile_intensity"),
                "confidence": group.get("confidence"),
                "review_flags_json": json_text(group.get("review_flags")),
                **geometry_fields(group),
            })

cases = pd.DataFrame(case_rows).sort_values(["image_index", "source_id"]).reset_index(drop=True)
ads = pd.DataFrame(ad_rows).sort_values(["image_index", "source_id", "ad_position"]).reset_index(drop=True)
people = pd.DataFrame(person_rows).sort_values(["image_index", "source_id", "ad_position", "person_position"]).reset_index(drop=True)
groups = pd.DataFrame(group_rows).sort_values(["image_index", "source_id", "ad_position", "group_position"]).reset_index(drop=True)

print({"cases": len(cases), "ads": len(ads), "people": len(people), "groups": len(groups)})

## Sanity checks and audit outputs

Hard invariants stop the notebook when the dataset is not the expected 300-case four-source panel or when relational keys are ambiguous. Content checks are written to an audit table rather than silently correcting judgments. This matters because a technically valid disagreement can still be substantively important.

In [ ]:
audit_rows: list[dict[str, Any]] = []


def audit(check: str, level: str, passed: bool, affected_rows: int, detail: str) -> None:
    audit_rows.append({
        "check": check,
        "level": level,
        "passed": bool(passed),
        "affected_rows": int(affected_rows),
        "detail": detail,
    })


audit("case_panel_size", "error", len(cases) == EXPECTED_CASES * 4, abs(len(cases) - EXPECTED_CASES * 4), f"expected {EXPECTED_CASES * 4}, observed {len(cases)}")
audit("case_primary_key_unique", "error", cases["case_key"].is_unique, int(cases["case_key"].duplicated().sum()), "source_id + image_id")
audit("ad_primary_key_unique", "error", ads["ad_key"].is_unique, int(ads["ad_key"].duplicated().sum()), "IDs are coder-local; source is part of the key")
audit("person_primary_key_unique", "error", people["person_key"].is_unique, int(people["person_key"].duplicated().sum()), "IDs are coder-local; source and parent ad are part of the key")
audit("group_primary_key_unique", "error", groups["group_key"].is_unique, int(groups["group_key"].duplicated().sum()), "IDs are coder-local; source and parent ad are part of the key")

ad_count_mismatch = cases["declared_qualifying_ad_count"].ne(cases["observed_ad_rows"])
audit("declared_ad_count_matches_rows", "error", not ad_count_mismatch.any(), int(ad_count_mismatch.sum()), "page declaration versus advertisements array")

orphan_people = ~people["ad_key"].isin(ads["ad_key"])
orphan_groups = ~groups["ad_key"].isin(ads["ad_key"])
audit("people_have_parent_ad", "error", not orphan_people.any(), int(orphan_people.sum()), "foreign-key check")
audit("groups_have_parent_ad", "error", not orphan_groups.any(), int(orphan_groups.sum()), "foreign-key check")

def invalid_present_boxes(frame: pd.DataFrame) -> pd.Series:
    present = frame["bbox_raw_json"].notna()
    valid = (
        frame["bbox_x1"].ge(0) & frame["bbox_y1"].ge(0)
        & frame["bbox_x2"].le(1) & frame["bbox_y2"].le(1)
        & frame["bbox_x2"].gt(frame["bbox_x1"])
        & frame["bbox_y2"].gt(frame["bbox_y1"])
    )
    return present & ~valid


for table_name, frame in [("ads", ads), ("people", people), ("groups", groups)]:
    invalid = invalid_present_boxes(frame)
    audit(f"{table_name}_boxes_valid_0_1", "error", not invalid.any(), int(invalid.sum()), "all present normalized boxes")

missing_person_box = people["bbox_raw_json"].isna() & ~people["is_duplicate_copy"]
audit("nonduplicate_people_have_boxes", "warning", not missing_person_box.any(), int(missing_person_box.sum()), "duplicate copies may intentionally omit geometry")

yes_without_intensity = people["smile_present"].eq("yes") & people["smile_intensity"].isna()
intensity_without_yes = ~people["smile_present"].eq("yes") & people["smile_intensity"].notna()
zero_legibility_with_expression = people["face_expression_legibility"].eq("0_not_legible") & (people["smile_present"].notna() | people["smile_intensity"].notna() | people["gaze_target"].notna())
audit("smile_yes_requires_intensity", "error", not yes_without_intensity.any(), int(yes_without_intensity.sum()), "conditional schema rule")
audit("smile_intensity_only_if_yes", "error", not intensity_without_yes.any(), int(intensity_without_yes.sum()), "conditional schema rule")
audit("zero_legibility_skips_expression", "error", not zero_legibility_with_expression.any(), int(zero_legibility_with_expression.sum()), "conditional schema rule")

audit_table = pd.DataFrame(audit_rows)
display(audit_table)

hard_failures = audit_table.query("level == 'error' and not passed")
assert hard_failures.empty, hard_failures.to_string(index=False)

entity_counts = pd.concat([
    cases.groupby(["source_id", "source_type"]).size().rename("cases"),
    ads.groupby(["source_id", "source_type"]).size().rename("ads"),
    people.groupby(["source_id", "source_type"]).size().rename("people"),
    groups.groupby(["source_id", "source_type"]).size().rename("groups"),
], axis=1).fillna(0).astype(int).reset_index()
display(entity_counts)

# Export the compact descriptive table used in Chapter 5. These are raw
# annotation volumes, not performance scores; entity matching happens later.
publication_entity_counts = (
    entity_counts
    .set_index("source_id")
    .loc[["A", "B", "C", "llm_qwen_final_standalone_v1"]]
    .reset_index()
    .assign(coder=lambda frame: frame["source_id"].map({
        "A": "A",
        "B": "B",
        "C": "C",
        "llm_qwen_final_standalone_v1": "LLM",
    }))
    [["coder", "ads", "people", "groups"]]
)

table_values_path, table_fragment_path = export_quarto_table(
    publication_entity_counts,
    "annotation-entity-counts",
    caption="Entities identified by each coder in the 300-case evaluation set.",
    label="tbl-annotation-entity-counts",
    column_labels={
        "coder": "Coder",
        "ads": "Advertisements",
        "people": "Individual faces",
        "groups": "Groups",
    },
    formats={"ads": ",d", "people": ",d", "groups": ",d"},
    note=(
        "Counts precede spatial matching and consensus construction. Differences "
        "therefore describe annotation volume, not detection accuracy."
    ),
)
display(publication_entity_counts)
print(f"Thesis table values:   {table_values_path}")
print(f"Quarto table fragment: {table_fragment_path}")


## Write and verify processed data

Parquet preserves nulls and numeric coordinates reliably and is efficient for later matching. CSV is used for the small human-readable audit and inventory tables. Every output is reloaded and checked before completion.

In [ ]:
frames = {"cases": cases, "ads": ads, "people": people, "groups": groups}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

parquet_outputs = {
    "cases": OUTPUT_DIR / "annotation_cases.parquet",
    "ads": OUTPUT_DIR / "annotation_ads.parquet",
    "people": OUTPUT_DIR / "annotation_people.parquet",
    "groups": OUTPUT_DIR / "annotation_groups.parquet",
}
for name, path in parquet_outputs.items():
    frames[name].to_parquet(path, index=False)

load_audit.to_csv(OUTPUT_DIR / "source_load_audit.csv", index=False)
audit_table.to_csv(OUTPUT_DIR / "preprocessing_checks.csv", index=False)
entity_counts.to_csv(OUTPUT_DIR / "entity_counts_by_source.csv", index=False)


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


metadata = {
    "dataset": "llm_evaluation_300",
    "expected_cases": EXPECTED_CASES,
    "target_image_ids_sha256": hashlib.sha256("\n".join(target_ids).encode("utf-8")).hexdigest(),
    "sources": [
        {
            "source_id": source_id,
            "source_type": "human",
            "path": str(path),
            "sha256": sha256(path),
        }
        for source_id, path in HUMAN_PATHS.items()
    ] + [{
        "source_id": "llm_qwen_final_standalone_v1",
        "source_type": "llm",
        "path": str(LLM_PATH),
        "sha256": sha256(LLM_PATH),
    }],
    "outputs": {name: path.name for name, path in parquet_outputs.items()},
    "normalizations": [
        "LLM advertisement_id exposed as ad_id; original_id_field records provenance",
        "bbox_1000 and face_bbox_1000 divided by 1000; raw JSON and scale retained",
        "depiction_type_resolved inherits the parent ad value only when person depiction_type is null",
        "no category labels are collapsed or made lenient in preprocessing",
    ],
}
(OUTPUT_DIR / "dataset_metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

reloaded = {name: pd.read_parquet(path) for name, path in parquet_outputs.items()}
for name, frame in frames.items():
    assert reloaded[name].shape == frame.shape, f"{name}: reload shape changed"
    key = {"cases": "case_key", "ads": "ad_key", "people": "person_key", "groups": "group_key"}[name]
    assert reloaded[name][key].is_unique

output_summary = pd.DataFrame([
    {"file": path.name, "rows": len(reloaded[name]), "columns": reloaded[name].shape[1], "bytes": path.stat().st_size}
    for name, path in parquet_outputs.items()
])
display(output_summary)
print("Processed evaluation data written and reloaded successfully.")